**<h1>Validation YOLOv3 VisDrone</h1>**

Evaluates a trained YOLO model across two dimensions:

**1. Detection Accuracy**
- mAP@50, mAP@50-95
- Precision, Recall
- Per-class mAP@50

**2. Inference Performance (GPU benchmark)**
- Latency per image (ms)
- FPS
- Warm-up corrected timing

> Results from this notebook are the **GPU baseline** for comparison against FPGA inference on the Kria KR260.

**Dependencies and GPU check**

In [ ]:
# !pip install ultralytics torch torchvision

import torch
import time
import numpy as np
from ultralytics import YOLO
import os

def check_device():
    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        vram = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"✅ GPU: {name}  |  VRAM: {vram:.2f} GB")
        return 0
    else:
        print("⚠️  No GPU detected — using CPU.")
        return "cpu"

DEVICE = check_device()

**<h3>Configuration</h3>**

**Important:** `IMG_SIZE` must match the `imgsz` used during training (default: 640).
Mismatched image sizes will produce artificially lower metrics.

In [ ]:
# ============================================================
# CONFIGURATION — update these paths before running
# ============================================================

MODEL_PATH = "runs/train_visdrone/YOUR_RUN_NAME/weights/best.pt"  # <-- update
DATA_YAML  = "./data/data.yaml"   # Must match OUTPUT_ROOT from training notebook

IMG_SIZE   = 640    # Must match imgsz used in training
BATCH_SIZE = 16     # Reduce to 8 if you get OOM
CONF_THRES = 0.25   # Confidence threshold for detections
IOU_THRES  = 0.5    # IoU threshold for NMS

# Latency benchmark config
WARMUP_RUNS  = 20   # Discarded runs to let GPU reach steady state
MEASURE_RUNS = 200  # Runs used for timing measurement

# ============================================================
print(f"Model  : {MODEL_PATH}")
print(f"Dataset: {DATA_YAML}")
print(f"Device : {DEVICE}")
print(f"ImgSize: {IMG_SIZE}")

# Verify paths
for path, label in [(MODEL_PATH, "Model"), (DATA_YAML, "data.yaml")]:
    status = "✅" if os.path.exists(path) else "❌ NOT FOUND"
    print(f"{status}  {label}: {path}")

**<h3>Accuracy Evaluation (mAP)</h3>**

Runs standard COCO-style evaluation on the validation split.
Generates confusion matrix and P/R curves automatically (`plots=True`).

In [ ]:
model = YOLO(MODEL_PATH)

results = model.val(
    data      = DATA_YAML,
    imgsz     = IMG_SIZE,
    batch     = BATCH_SIZE,
    conf      = CONF_THRES,
    iou       = IOU_THRES,
    device    = DEVICE,
    split     = "val",     # Explicit: always evaluate on val split
    save_json = True,      # Saves COCO-format results for external benchmarking
    plots     = True,      # Confusion matrix, P/R curves, F1 curves
    verbose   = True,
)

# ── Global metrics ──────────────────────────────────────────
print("\n" + "=" * 50)
print("  ACCURACY RESULTS")
print("=" * 50)
print(f"  mAP@50        : {results.box.map50:.4f}")
print(f"  mAP@50-95     : {results.box.map:.4f}")
print(f"  Precision (P) : {results.box.mp:.4f}")
print(f"  Recall    (R) : {results.box.mr:.4f}")

# ── Per-class mAP@50 ────────────────────────────────────────
print("\n  mAP@50 per class:")
print(f"  {'Class':<22} {'AP@50':>8}")
print("  " + "-" * 32)
names = model.names
for i, ap in enumerate(results.box.ap50):
    flag = "  " if ap >= 0.3 else "⚠️"   # flag weak classes
    print(f"  {flag} {names[i]:<20} {ap:.4f}")

print("\nPlots saved to:", results.save_dir)

**<h3>Inference Latency Benchmark (GPU)</h3>**

Measures real inference latency on the GPU.
This is the **baseline** you will compare against the Kria KR260 FPGA.

**Method:**
- Warm-up phase: discard first N runs (GPU thermal/clock ramp-up)
- Measurement phase: time N runs, compute mean ± std
- Uses `torch.cuda.synchronize()` for accurate GPU timing
- Input: random tensor simulating a real image batch

In [ ]:
import torch
import numpy as np

model = YOLO(MODEL_PATH)
model_torch = model.model.eval()

if DEVICE != "cpu":
    model_torch = model_torch.cuda()

# Dummy input — single image, same size as training
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
if DEVICE != "cpu":
    dummy = dummy.cuda()

print(f"Running latency benchmark: {WARMUP_RUNS} warm-up + {MEASURE_RUNS} measured runs")
print(f"Input shape: {list(dummy.shape)}")

# ── Warm-up phase ───────────────────────────────────────────
# GPU needs a few runs to reach steady-state clock speed.
# Warm-up results are discarded.
with torch.no_grad():
    for _ in range(WARMUP_RUNS):
        _ = model_torch(dummy)
        if DEVICE != "cpu":
            torch.cuda.synchronize()

# ── Measurement phase ───────────────────────────────────────
latencies_ms = []

with torch.no_grad():
    for _ in range(MEASURE_RUNS):
        if DEVICE != "cpu":
            torch.cuda.synchronize()
        t0 = time.perf_counter()

        _ = model_torch(dummy)

        if DEVICE != "cpu":
            torch.cuda.synchronize()  # Wait for GPU to finish before stopping timer
        t1 = time.perf_counter()

        latencies_ms.append((t1 - t0) * 1000)

# ── Results ─────────────────────────────────────────────────
lat = np.array(latencies_ms)

mean_ms = lat.mean()
std_ms  = lat.std()
p50_ms  = np.percentile(lat, 50)
p95_ms  = np.percentile(lat, 95)
p99_ms  = np.percentile(lat, 99)
fps     = 1000.0 / mean_ms

gpu_name = torch.cuda.get_device_name(0) if DEVICE != "cpu" else "CPU"

print("\n" + "=" * 50)
print("  INFERENCE LATENCY — GPU BASELINE")
print("=" * 50)
print(f"  Device          : {gpu_name}")
print(f"  Model           : YOLOv3  |  imgsz: {IMG_SIZE}")
print(f"  Batch size      : 1 (single image)")
print(f"  Runs measured   : {MEASURE_RUNS}")
print()
print(f"  Mean latency    : {mean_ms:.2f} ms")
print(f"  Std dev         : {std_ms:.2f} ms")
print(f"  Median (p50)    : {p50_ms:.2f} ms")
print(f"  p95 latency     : {p95_ms:.2f} ms")
print(f"  p99 latency     : {p99_ms:.2f} ms")
print(f"  Throughput      : {fps:.1f} FPS")
print()
print("  → Save these values for comparison against Kria KR260 FPGA inference.")

**<h3>GPU vs FPGA Comparison Table</h3>**

Fill in the FPGA numbers manually after running inference on the Kria KR260.
This cell formats the final comparison table for your report.

In [ ]:
# ── Fill these after running on the Kria KR260 ──────────────
FPGA_LATENCY_MS = None   # e.g. 45.3
FPGA_FPS        = None   # e.g. 22.1
FPGA_MAP50      = None   # e.g. 0.312  (should match GPU — same model weights)

# ── GPU values from Cell 4 ──────────────────────────────────
GPU_LATENCY_MS  = mean_ms
GPU_FPS         = fps
GPU_MAP50       = results.box.map50

print("=" * 60)
print(f"  {'Metric':<22} {'GPU (RTX 2000)':>16} {'FPGA (KR260)':>16}")
print("=" * 60)

def fmt(val, unit="", decimals=2):
    if val is None:
        return "-- (pending)"
    return f"{val:.{decimals}f}{unit}"

rows = [
    ("mAP@50",          fmt(GPU_MAP50, decimals=4),       fmt(FPGA_MAP50, decimals=4)),
    ("Latency (ms)",    fmt(GPU_LATENCY_MS, " ms"),       fmt(FPGA_LATENCY_MS, " ms")),
    ("Throughput (FPS)",fmt(GPU_FPS, " FPS", 1),          fmt(FPGA_FPS, " FPS", 1)),
    ("Power (W)",       "~70 W (TDP est.)",               "~5–10 W (KR260)"),
]

for label, gpu_val, fpga_val in rows:
    print(f"  {label:<22} {gpu_val:>16} {fpga_val:>16}")

print("=" * 60)
print()

# Efficiency ratio (FPS per watt) — only if FPGA numbers are available
if FPGA_FPS is not None:
    gpu_eff  = GPU_FPS / 70     # RTX 2000 Ada ~70W TDP
    fpga_eff = FPGA_FPS / 7.5   # KR260 ~7.5W typical
    print(f"  GPU efficiency  : {gpu_eff:.2f} FPS/W")
    print(f"  FPGA efficiency : {fpga_eff:.2f} FPS/W")
    print(f"  FPGA efficiency gain: {fpga_eff / gpu_eff:.1f}x")